## Finetuned agent — inference test

**Tool tokens** must match training (`src/training/trainer.py`): load **`tokenizer_config.json`** from the **same folder** as the LoRA adapter (`adapter_config.json`).

- **This notebook:** `./models/phi3-agent-final` (final adapter + tokenizer).

Run the code cell below, then the **Run tests** cell.

In [1]:
"""
Test Phi-3.5 + LoRA agent (SFT and/or RL). Tokenizer MUST come from the adapter save dir
so added tool tokens match training IDs.
"""
import logging
from pathlib import Path

import torch
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer
from transformers.utils import logging as hf_logging
import transformers.modeling_utils as modeling_utils

if not hasattr(modeling_utils, "shard_checkpoint"):
    def shard_checkpoint(*args, **kwargs):
        # simple fallback: return full state dict without sharding
        return args[0], None

    modeling_utils.shard_checkpoint = shard_checkpoint
    

def _patch_dynamic_cache_for_phi3_hub() -> None:
    """Hub Phi-3 vs Transformers 4.48+: restore seen_tokens, get_max_length, get_usable_length."""
    try:
        from transformers.cache_utils import DynamicCache

        if not hasattr(DynamicCache, "seen_tokens"):
            DynamicCache.seen_tokens = property(lambda self: self.get_seq_length(0))
        if not hasattr(DynamicCache, "get_max_length"):

            def get_max_length(self, layer_idx: int = 0) -> int:
                return self.get_max_cache_shape(layer_idx)

            DynamicCache.get_max_length = get_max_length
        if not hasattr(DynamicCache, "get_usable_length"):

            def get_usable_length(self, new_seq_length: int, layer_idx: int = 0) -> int:
                max_length = self.get_max_cache_shape(layer_idx)
                prev = self.get_seq_length(layer_idx)
                if max_length is not None and max_length > 0 and prev + new_seq_length > max_length:
                    return max_length - new_seq_length
                return prev

            DynamicCache.get_usable_length = get_usable_length
    except Exception as ex:
        logging.warning("DynamicCache hub-compat patch skipped: %s", ex)


logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s")
hf_logging.set_verbosity_error()

_patch_dynamic_cache_for_phi3_hub()

BASE_MODEL = "microsoft/Phi-3.5-mini-instruct"

# Final model (adapter + tokenizer): models/phi3-agent-final
ADAPTER_PATH = "./models/phi3-agent-final"

# Checkpoint fallbacks disabled — using final export only.
# ADAPTER_FALLBACKS = [
#     "./results/checkpoint-1000",
#     "./results/checkpoint-500",
#     "./results_rl/checkpoint-rl-150",
#     "./results_rl/checkpoint-rl-100",
#     "./results_rl/checkpoint-rl-50",
# ]
ADAPTER_FALLBACKS: list[str] = []

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE = torch.bfloat16 if DEVICE == "cuda" and torch.cuda.is_bf16_supported() else torch.float16

# Must stay in sync with src/training/trainer.py TOOL_SPECIAL_TOKENS
TOOL_SPECIAL_TOKENS = [
    "<tool_use>",
    "</tool_use>",
    "<tool_name>",
    "</tool_name>",
    "<parameters>",
    "</parameters>",
]


def verify_tool_special_tokens(tokenizer: AutoTokenizer) -> dict[str, int]:
    """Ensure each tool string is exactly one subword id (matches training tokenizer)."""
    unk = getattr(tokenizer, "unk_token_id", None)
    mapping: dict[str, int] = {}
    for t in TOOL_SPECIAL_TOKENS:
        ids = tokenizer.encode(t, add_special_tokens=False)
        if len(ids) != 1:
            raise ValueError(
                f"Token {t!r} encodes to {len(ids)} ids (expected 1). "
                "Load tokenizer from the adapter folder saved after finetuning, not base Phi-3 only."
            )
        tid = ids[0]
        if unk is not None and tid == unk:
            raise ValueError(
                f"Token {t!r} maps to UNK. Use tokenizer files from your training/RL output directory."
            )
        mapping[t] = tid
    added = getattr(tokenizer, "added_tokens_encoder", {}) or {}
    for t, tid in mapping.items():
        if t in added and added[t] != tid:
            logging.warning("Mismatch added_tokens_encoder for %s", t)
    logging.info("Tool special tokens OK (%d tokens, vocab size %s)", len(mapping), len(tokenizer))
    return mapping


def resolve_adapter_dir() -> Path:
    candidates = [Path(ADAPTER_PATH)] + [Path(p) for p in ADAPTER_FALLBACKS]
    seen: set[str] = set()
    uniq: list[Path] = []
    for c in candidates:
        key = str(c.resolve()) if c.exists() else str(c)
        if key in seen:
            continue
        seen.add(key)
        uniq.append(c)
    for c in uniq:
        if (c / "adapter_config.json").is_file() and (c / "tokenizer_config.json").is_file():
            logging.info("Using adapter: %s", c.resolve())
            return c
    tried = ", ".join(str(c) for c in uniq)
    raise FileNotFoundError(
        "No adapter directory with both adapter_config.json and tokenizer_config.json. Tried: " + tried
    )


def load_tokenizer(adapter_dir: Path) -> AutoTokenizer:
    tok = AutoTokenizer.from_pretrained(str(adapter_dir), trust_remote_code=True)
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token
    verify_tool_special_tokens(tok)
    for t in TOOL_SPECIAL_TOKENS:
        logging.info("%s -> id %s", t, tok.convert_tokens_to_ids(t))
    return tok


def load_model(adapter_dir: Path | None = None, merge_lora: bool = False):
    root = adapter_dir or resolve_adapter_dir()
    tokenizer = load_tokenizer(root)

    attn_impl = "sdpa" if DEVICE == "cuda" else "eager"
    try:
        base_model = AutoModelForCausalLM.from_pretrained(
            BASE_MODEL,
            torch_dtype=DTYPE if DEVICE == "cuda" else torch.float32,
            trust_remote_code=True,
            attn_implementation=attn_impl,
        ).to(DEVICE)
    except (ValueError, TypeError) as e:
        logging.warning("attn_implementation=%s failed (%s); retrying eager.", attn_impl, e)
        base_model = AutoModelForCausalLM.from_pretrained(
            BASE_MODEL,
            torch_dtype=DTYPE if DEVICE == "cuda" else torch.float32,
            trust_remote_code=True,
            attn_implementation="eager",
        ).to(DEVICE)

    if len(tokenizer) != base_model.get_input_embeddings().weight.shape[0]:
        base_model.resize_token_embeddings(len(tokenizer))

    model = PeftModel.from_pretrained(base_model, str(root))
    model.eval()

    if merge_lora:
        model = model.merge_and_unload()
        logging.info("LoRA merged into base weights.")

    logging.info("Model ready (vocab %s).", len(tokenizer))
    return model, tokenizer


def build_prompt(user_input: str) -> str:
    return f"""### Instruction:
{user_input}

### Response:
"""


def generate(
    model,
    tokenizer,
    text: str,
    max_new_tokens: int = 384,
    greedy: bool = False,
    temperature: float = 0.7,
    top_p: float = 0.9,
):
    """Decode only the continuation after the prompt (keeps tool tags visible)."""
    prompt = build_prompt(text)
    inputs = tokenizer(prompt, return_tensors="pt", add_special_tokens=True).to(DEVICE)
    prompt_len = inputs["input_ids"].shape[1]

    gen_kwargs = dict(
        max_new_tokens=max_new_tokens,
        pad_token_id=tokenizer.pad_token_id,
        eos_token_id=tokenizer.eos_token_id,
        use_cache=True,
    )
    if greedy:
        gen_kwargs["do_sample"] = False
    else:
        gen_kwargs["do_sample"] = True
        gen_kwargs["temperature"] = max(0.01, temperature)
        gen_kwargs["top_p"] = top_p

    with torch.no_grad():
        out = model.generate(**inputs, **gen_kwargs)

    new_tokens = out[0, prompt_len:]
    return tokenizer.decode(new_tokens, skip_special_tokens=False).strip()


def run_tests():
    adapter_root = resolve_adapter_dir()
    model, tokenizer = load_model(adapter_root, merge_lora=False)

    test_queries = [
        "Help me with: summarize latest AI trends",
        "Help me with: read a configuration file and summarize it",
        "Help me with: compare cost of living in Bangalore and Tokyo",
    ]

    for q in test_queries:
        print("\n" + "=" * 60)
        print("User:", q)
        try:
            response = generate(model, tokenizer, q, greedy=True, max_new_tokens=512)
            print("Agent (greedy):\n", response)
        except Exception as e:
            print("Error:", e)


def chat():
    model, tokenizer = load_model(resolve_adapter_dir())
    print("Chat (type exit or quit)\n")
    while True:
        user_input = input("You: ")
        if user_input.lower() in ("exit", "quit"):
            break
        try:
            reply = generate(model, tokenizer, user_input, greedy=False)
            print("Agent:\n", reply, "\n", "-" * 50, sep="")
        except Exception as e:
            print("Error:", e)


if __name__ == "__main__":
    run_tests()


c:\Users\SARANSH\Anaconda3\envs\llm_env\Lib\site-packages\torch\cuda\__init__.py:216: UserWarning: expandable_segments not supported on this platform (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\c10/cuda/CUDAAllocatorConfig.h:39.)
  torch.tensor([1.0], dtype=torch.bfloat16, device=device)
2026-03-23 06:09:31,645 - INFO - Using adapter: D:\self projects\FileWise-Agent\MCP\SLM_Agent\SLM_Agent_Github\slm-agent\models\phi3-agent-final
2026-03-23 06:09:31,976 - INFO - Tool special tokens OK (6 tokens, vocab size 32017)
2026-03-23 06:09:31,976 - INFO - <tool_use> -> id 32011
2026-03-23 06:09:31,977 - INFO - </tool_use> -> id 32012
2026-03-23 06:09:31,977 - INFO - <tool_name> -> id 32013
2026-03-23 06:09:31,978 - INFO - </tool_name> -> id 32014
2026-03-23 06:09:31,979 - INFO - <parameters> -> id 32015
2026-03-23 06:09:31,979 - INFO - </parameters> -> id 32016
2026-03-23 06:09:32,301 - INFO - HTTP Request: HEAD https://huggingface.co/microsoft/Phi-3.5-mini-instruct/

Loading weights:   0%|          | 0/195 [00:00<?, ?it/s]

2026-03-23 06:09:34,642 - INFO - HTTP Request: HEAD https://huggingface.co/microsoft/Phi-3.5-mini-instruct/resolve/main/generation_config.json "HTTP/1.1 307 Temporary Redirect"
2026-03-23 06:09:34,652 - INFO - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/microsoft/Phi-3.5-mini-instruct/2fe192450127e6a83f7441aef6e3ca586c338b77/generation_config.json "HTTP/1.1 200 OK"
2026-03-23 06:09:34,879 - INFO - HTTP Request: HEAD https://huggingface.co/microsoft/Phi-3.5-mini-instruct/resolve/main/custom_generate/generate.py "HTTP/1.1 404 Not Found"
c:\Users\SARANSH\Anaconda3\envs\llm_env\Lib\site-packages\awq\modules\linear\exllama.py:12: UserWarning: AutoAWQ could not load ExLlama kernels extension. Details: DLL load failed while importing exl_ext: The specified procedure could not be found.
  warnings.warn(f"AutoAWQ could not load ExLlama kernels extension. Details: {ex}")
c:\Users\SARANSH\Anaconda3\envs\llm_env\Lib\site-packages\awq\modules\linear\exllamav2.py:13: UserWarni


User: Help me with: summarize latest AI trends
Error: 'DynamicCache' object has no attribute 'get_usable_length'

User: Help me with: read a configuration file and summarize it
Error: 'DynamicCache' object has no attribute 'get_usable_length'

User: Help me with: compare cost of living in Bangalore and Tokyo
Error: 'DynamicCache' object has no attribute 'get_usable_length'
